In [2]:
import os 
from dotenv import load_dotenv
from google import genai

load_dotenv()
client =  genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Ollama locally?")

Tokens - Input: 791, Output: 365


'To run Ollama locally, follow these steps based on the provided context:\n\n### 1. Install Ollama\nVisit [https://ollama.com/download](https://ollama.com/download) and choose the installation method for your operating system:\n* **macOS**: Download and install the `.pkg` file.\n* **Windows**: Download and install the `.msi` file.\n* **Linux**: Run the following command in your terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n### 2. Run the Model\nOnce installed, open a terminal and run the following command to download the LLaMA 3 model, start it locally, and open a chat interface:\n```bash\nollama run llama3\n```\n\n### 3. Test the Local Server\nTo verify that the Ollama local server is running, run:\n```bash\ncurl http://localhost:11434\n```\nYou should receive a JSON response listing the models.\n\n### 4. Optional: Run via Python\nYou can install the Python client and run the model in a script:\n```bash\npip install ollama\n```\nMinimal Python example

In [6]:
assistant.rag("How do I run Olama locally?")

Tokens - Input: 934, Output: 18


'Based on the provided context, there is no information on how to run Olama locally.'

In [7]:
from google import genai
from google.genai import types

messages = [
    types.Content(
        role="user", 
        parts=[types.Part.from_text(text="How do I run Olama?")]
    )
]

In [8]:
def search(query: str) -> dict:
    """Search the FAQ database for entries matching the given query."""
    
    # We are simulating the RAG index search here
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "query": types.Schema(
                type=types.Type.STRING,
                description="Search query text to look up in the course FAQ."
            )
        },
        required=["query"]
    )
)

# 2. Wrap it in a Tool object
search_tool = types.Tool(
    function_declarations=[search_declaration]
)

In [10]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[search_tool], # Gemini automatically parses the function!
    )
)

# Check if the model decided to use a tool
if response.function_calls:
    call = response.function_calls[0]
    print(f"Function requested: {call.name}")
    print(f"Arguments generated: {call.args}")

Function requested: search
Arguments generated: {'query': 'How do I run Olama?'}


In [11]:
import json

if response.function_calls:
    call = response.function_calls[0]
    
    # 1. Execute the actual Python function using the arguments the LLM provided
    # (In a production agent with multiple tools, you'd route this based on call.name)
    tool_results = search(**call.args)
    
    # 2. Append the model's function call request to our conversation history
    # The model needs to remember that *it* made this request
    messages.append(response.candidates[0].content)
    
    # 3. Append the execution result back to the model
    # We wrap the result in a FunctionResponse part
    messages.append(
        types.Content(
            role="user",
            parts=[
                types.Part.from_function_response(
                    name=call.name,
                    response={"results": tool_results}
                )
            ]
        )
    )

In [12]:
final_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config=types.GenerateContentConfig(tools=[search])
)

print(final_response.text)

I couldn't find any information about "Olama" in my database. It's possible that it's not a topic covered, or there might be a typo in the name. Can you provide more context or clarify the name?


In [13]:
usage = final_response.usage_metadata

input_tokens = usage.prompt_token_count
output_tokens = usage.candidates_token_count

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")

Input tokens: 1163
Output tokens: 49


## Agentic Loop

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [15]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [16]:
question = "I just discovered the course. Can I join it?"

# 1. Set up the initial memory
messages = [
    types.Content(role="user", parts=[types.Part.from_text(text=question)])
]

# 2. Configure the instructions and tools
config = types.GenerateContentConfig(
    system_instruction=instructions,
    tools=[search]
)

# 3. Call the model once
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config=config,
)

# 4. Append the model's output to the history so it remembers its own actions
if response.candidates and response.candidates[0].content:
    messages.append(response.candidates[0].content)

has_function_calls = False

print("MESSAGES: ",messages)

# 5. Process any function calls
if response.function_calls:
    has_function_calls = True
    tool_responses = []
    
    for fc in response.function_calls:
        print("function_call:", fc.name, fc.args)

        call_output = make_call(fc)
        tool_responses.append(call_output)
    
    # Append the tool's result back to the history as a 'user' message
    messages.append(
        types.Content(role="user", parts=tool_responses)
    )

# 6. Print any text the assistant provided alongside the tool call
if response.text:
    print("ASSISTANT:")
    print(response.text)

MESSAGES:  [Content(
  parts=[
    Part(
      text='I just discovered the course. Can I join it?'
    ),
  ],
  role='user'
), Content(
  parts=[
    Part(
      text="""Yes, you can still join the course. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.

Are there other areas you'd like to explore?""",
      thought_signature=b'\n\xf7\x02\x01\x0c9\xd6\xc72/\x8e\xdaE\x1d\x0fr\x97\xa8\xad\x82\x94\xe9\xd5\xb2\xaf\xf9a\xdc\xa3\x0f\r\xe3\xcb\xa0\xd6\xc6s\xf3\xc4\x19c?\xac{\x99\x1cK^w\xbaG\xe5o\xafC\xf8\x97\xa1(W\xecE_\t)\x19/Gx\x9b\xdc~F\xdd\xd3S@E\n\x9f\xa6\x8eG\x8ej\rj\xb7,TU\xf9&:}f*...'
    ),
  ],
  role='model'
)]
ASSISTANT:
Yes, you can still join the course. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.

Are there other areas you'd like to explore?


In [17]:
def agent_loop(instructions: str, question: str, model="gemini-2.5-flash") -> str:
    # 3. MEMORY: Initialize the conversation history with the user's question
    messages = [
        types.Content(role="user", parts=[types.Part.from_text(text=question)])
    ]

    # Setup the configuration with our instructions and tools
    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[search], 
        temperature=0.2 # Lower temperature for more analytical tool use
    )

    it = 1
    last_answer = ""

    while True:
        print(f"\n--- iteration #{it} ---")
        has_function_calls = False

        # Call the model with our current memory and config
        response = client.models.generate_content(
            model=model,
            contents=messages,
            config=config
        )

        # Append the model's entire response (which could contain text, tool calls, or both) to memory
        if response.candidates and response.candidates[0].content:
            messages.append(response.candidates[0].content)

        # Check if the model decided to call a function
        if response.function_calls:
            has_function_calls = True
            
            # Prepare a list to hold the results of all tool calls made in this turn
            tool_responses = []

            print(response.function_calls)

            for fc in response.function_calls:
                print(f"function_call: {fc.name} {fc.args}")
                
                # The Function-Call Helper logic
                if fc.name == "search":
                    # The SDK parses the JSON arguments into a Python dict for us
                    result = search(**fc.args) 
                    
                    # Package the result into the format Gemini expects
                    tool_responses.append(
                        types.Part.from_function_response(
                            name=fc.name,
                            response=result
                        )
                    )
            
            # Append the tool results back to memory so the model sees them on the next loop.
            # In Gemini, function responses are appended under the "user" role.
            messages.append(
                types.Content(role="user", parts=tool_responses)
            )

        # Print any text the assistant generated during this turn
        if response.text:
            print("ASSISTANT:")
            print(response.text)
            last_answer = response.text

        it += 1
        
        # Exit condition: If the model didn't ask to call any functions, the loop is done.
        if not has_function_calls:
            break

    return last_answer

In [18]:
agent_loop(instructions, "When can I join the course")


--- iteration #1 ---
ASSISTANT:
You can join the course whenever you want, as the videos and GitHub materials are always available. However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted.

Are there any other areas you'd like to explore?


"You can join the course whenever you want, as the videos and GitHub materials are always available. However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted.\n\nAre there any other areas you'd like to explore?"

In [19]:
agent_loop(instructions, "what's queen gambit?")


--- iteration #1 ---
ASSISTANT:
I can only answer questions about the course. Is there anything about the course or its logistics that you'd like to know?


"I can only answer questions about the course. Is there anything about the course or its logistics that you'd like to know?"

In [20]:
from langchain_core.tools import tool

@tool
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )


In [21]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=instructions,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "When can I join the course?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'Yes, you can still join the course even if you just discovered it. You can start learning and submitting homework without formal registration, as registration is primarily to gauge', 'extras': {'signature': 'CnIBDDnWxxindwn36PqU5FfF7pU8omebPqPq7w1aK1zX/oqwz+C7dMPjb1PsWbQcvgNUc9ZP9TeV0nd6wAgoUaaIDztKxzm+07ndmaMlByAZm6UiMRm3YakrCR0+GEdqFNCzYvuCGUR/Z0rP2B66UlizId4KwwEBDDnWx7gxDZvM9nG2fjzUT0hfBScFCFegog6PUabmRDXA/rvZqrhz78h+8Fzg1uZ6rbT/qJPdi6aiNU6JnsBPP/dxAqvl5SKbrbRZiNFw+4juG4HTE86L0FJAIyU6cbG3qKKWBqUgFISnR9Jur3WKYPLal01dDRk5E+azZCGgagGNCxCdSRr7o7/l7RrhdYF3gTYQF1nRlzU0RyuS60iqKqe047Bkl/of82QsJKBJcJhky7ucpsgLTOhn4JADGxozc1gKhAIBDDnWx07xrDCjdEFhAsDmosqd5AWs759gZ7OZ8qz32SzLojNZlTRNEC7Cy4UynHr+B2BJmlsU/fZyWlZdllC58vOkSTgXKyhx2zPUR35rNTl50dxbUQrxUzYFZIfUuuG8Oksz8M6VCWbJOxpoVWGQu83yXkaZkYZAMPBIpVvv/TEOuSiqryw6sn2ZODRNFVy5NpsYmQJ+rkqOxfQS4dVVIj3Pgh01q6G4ffbXNHi4ThzwM3QutLHSOBvqMcuVEPCPwJZAF7VeiFUgra6XqAwVCbNSbjz2mFubG4NqNVlZbCdMyBirvscg9W6z9jsQ/Z5CyM3KmDM8atjsD0ULnEnGo